# 959. Regions Cut By Slashes

## Topic Alignment
- **Role Relevance**: Model spatial partitioning problems such as floor plan analysis, network segmentation, or geographic region division.
- **Scenario**: Count isolated regions in maps with diagonal boundaries, useful in image segmentation or layout analysis.

## Metadata Summary
- Source: [Regions Cut By Slashes](https://leetcode.com/problems/regions-cut-by-slashes/)
- Tags: `Union-Find`, `Graph`, `Grid`, `DFS`
- Difficulty: Medium
- Recommended Priority: High

## Problem Statement
An `n x n` grid is composed of `1 x 1` squares where each `1 x 1` square consists of a `/`, `\\`, or blank space `' '`. These characters divide the square into contiguous regions.

Given the grid `grid` represented as a string array, return the number of regions.

Note that backslash characters are escaped, so a `\\` is represented as `'\\\\'`.

## Progressive Hints
- Hint 1: Subdivide each cell into 4 triangular sub-regions (top, right, bottom, left).
- Hint 2: Connect adjacent sub-regions based on the character in the cell.
- Hint 3: '/' connects top-left and bottom-right; '\\' connects top-right and bottom-left; ' ' connects all four.
- Hint 4: Use Union-Find to merge connected sub-regions and count components.

## Solution Overview
Subdivide each cell into 4 triangular regions. Union regions based on slashes and adjacency. Count the final number of connected components using Union-Find.

## Detailed Explanation
1. **Subdivision**: Each cell (i, j) has 4 sub-regions indexed 0-3 (top, right, bottom, left).
2. **Internal connections**:
   - '/': connect top(0) with left(3), right(1) with bottom(2)
   - '\\': connect top(0) with right(1), left(3) with bottom(2)
   - ' ': connect all four regions (0-1-2-3)
3. **External connections**: Connect adjacent cells:
   - Right side of cell (i,j) connects to left side of cell (i, j+1)
   - Bottom side of cell (i,j) connects to top side of cell (i+1, j)
4. **Indexing**: Use `(i * n + j) * 4 + region` to uniquely identify each sub-region.
5. **Count**: Number of regions = number of connected components in Union-Find.

## Complexity Trade-off Table
| Approach | Time Complexity | Space Complexity | Notes |
| --- | --- | --- | --- |
| Union-Find with subdivision | O(n² α(n)) | O(n²) | Efficient for large grids |
| DFS on upscaled grid | O(n²) | O(n²) | Upscale to 3n x 3n grid |
| BFS on upscaled grid | O(n²) | O(n²) | Similar to DFS |

## Reference Implementation

In [ ]:
from typing import List


class UnionFind:
    def __init__(self, n: int):
        self.parent = list(range(n))
        self.components = n
    
    def find(self, x: int) -> int:
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])  # Path compression
        return self.parent[x]
    
    def union(self, x: int, y: int):
        root_x, root_y = self.find(x), self.find(y)
        if root_x != root_y:
            self.parent[root_y] = root_x
            self.components -= 1
    
    def get_components(self) -> int:
        return self.components


def regionsBySlashes(grid: List[str]) -> int:
    n = len(grid)
    # Each cell has 4 sub-regions: 0=top, 1=right, 2=bottom, 3=left
    uf = UnionFind(4 * n * n)
    
    def get_index(row: int, col: int, region: int) -> int:
        return (row * n + col) * 4 + region
    
    for i in range(n):
        for j in range(n):
            char = grid[i][j]
            base = get_index(i, j, 0)
            
            # Internal connections within the cell
            if char == '/':
                uf.union(base + 0, base + 3)  # top with left
                uf.union(base + 1, base + 2)  # right with bottom
            elif char == '\\\\':
                uf.union(base + 0, base + 1)  # top with right
                uf.union(base + 2, base + 3)  # bottom with left
            else:  # ' '
                uf.union(base + 0, base + 1)
                uf.union(base + 1, base + 2)
                uf.union(base + 2, base + 3)
            
            # External connections to adjacent cells
            # Connect to right neighbor
            if j + 1 < n:
                uf.union(get_index(i, j, 1), get_index(i, j + 1, 3))
            
            # Connect to bottom neighbor
            if i + 1 < n:
                uf.union(get_index(i, j, 2), get_index(i + 1, j, 0))
    
    return uf.get_components()

## Validation

In [ ]:
assert regionsBySlashes([" /","/ "]) == 2
assert regionsBySlashes([" /","  "]) == 1
assert regionsBySlashes(["/\\\\","\\\\/"]) == 5
assert regionsBySlashes(["//","/ "]) == 3
assert regionsBySlashes([" "]) == 1
print('All tests passed for LC 959.')

## Complexity Analysis
- Time Complexity: O(n² α(n)), where n is the grid size and α(n) is the inverse Ackermann function.
- Space Complexity: O(n²) for storing 4n² sub-regions in Union-Find.
- Bottleneck: Processing each cell and its connections.

## Edge Cases & Pitfalls
- Single cell: Depends on character ('/' = 2 regions, ' ' = 1 region).
- All empty spaces: Should return 1 (entire grid is one region).
- Escape sequences: Remember '\\' is represented as '\\\\' in Python strings.
- Boundary connections: Carefully handle grid edges.

## Follow-up Variants
- Extend to 3D grids with plane divisions.
- Find the largest region instead of counting all regions.
- Dynamic updates: add/remove slashes and update region count.

## Takeaways
- Subdivision technique transforms complex geometry into graph connectivity.
- Union-Find naturally handles region merging and counting.
- Careful indexing is crucial for tracking sub-regions.
- Alternative: upscaling to 3x finer grid, but Union-Find approach is more elegant.

## Similar Problems
| Problem ID | Problem Title | Technique |
| --- | --- | --- |
| 200 | Number of Islands | Union-Find on grid |
| 305 | Number of Islands II | Dynamic Union-Find |
| 130 | Surrounded Regions | Union-Find with boundary |